# Two-Stage Position Model v3

**v2 대비 변경사항**:
1. 핵심 로직 모듈화 (`3_modeling/modules/`)
2. 다중 모델 지원 (LightGBM, XGBoost, CatBoost, RF, ExtraTrees)
3. 이상치 처리 방법 선택 (winsorize, iqr_clip, grubbs, lot_local)
4. Stage 1 클래스 불균형 처리 (scale_pos_weight, SMOTE)
5. Stage 2 Loss Function 실험 (regression, poisson, tweedie)
6. CLF Threshold 튜닝 (RMSE 기준 최적 탐색)
7. 다중 모델 비교 (Stage 7-A)
8. HPO (Optuna)
9. 보팅 / 스태킹 앙상블
10. 실험 파라미터 중복 검사


## 1. 환경 설정 및 데이터 로드

In [ ]:
# ============================================================
# 환경 설정 + 데이터 로드
# ============================================================
import os, sys

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/model_zoo.py'):
        os.system('gdown --id 1YVlGm6pitmPVOgTB-T4hcRNwQFhGjgfM -O /content/modeling.zip')
        os.system('unzip -qo /content/modeling.zip -d /content/project')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# --- 기본 라이브러리 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- 프로젝트 유틸 ---
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, POSITION_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import evaluate, postprocess, compare_models
from utils.experiment import (
    log_experiment, check_exp_id, check_duplicate_params, download_from_drive,
)

# --- 전처리 모듈 (2_preprocessing/) ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '2_preprocessing'))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from feature_selection import run_feature_selection, DEVICE

# --- 모델링 모듈 (3_modeling/modules/) ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '3_modeling'))
from modules.model_zoo import get_default_params, AVAILABLE_MODELS, DEVICE as MODEL_DEVICE
from modules.training import (
    run_classification, run_twostage_regression, run_single_regression,
    run_multi_model_comparison,
)

# --- 데이터 로드 ---
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys['train']

print(f'Device: {MODEL_DEVICE}')
print(f'Available models: {AVAILABLE_MODELS}')


In [ ]:
# ============================================================
# 실험 설정 셀 — 모든 파라미터를 여기서 제어
# ============================================================

EXP_ID = '3-1-004'
EXP_TYPE = 'Two-Stage v3 (modularized + multi-model)'
EXP_MEMO = 'v3: 모듈화, 다중 모델, 불균형 처리, threshold 튜닝'

# --- Google Drive 파일 ID ---
XLSX_GDRIVE_ID = '1IgaNh7ixgqpmH5PiwmSFbK2li6GODdew'
JSON_GDRIVE_ID = '1ycr6n5Ty_jzl4F-qQE4Cv5nS2WbIAZih'

# ═══════════════════════════════════════════════════════════════
# [레고 블록] 각 단계 ON/OFF 스위치
# ═══════════════════════════════════════════════════════════════
RUN_MULTI_MODEL      = True    # 6.5 다중 모델 비교 (Stage 7-A)
RUN_THRESHOLD_TUNING = True    # 7.  CLF Threshold 자동 튜닝
RUN_HPO              = False   # 7.5 Optuna HPO (시간 오래 걸림)
RUN_VOTING           = True    # 8.  보팅 앙상블
RUN_STACKING         = True    # 8.  스태킹 앙상블
RUN_ENSEMBLE         = RUN_VOTING or RUN_STACKING

# ═══════════════════════════════════════════════════════════════
# [전처리] 클리닝
# ═══════════════════════════════════════════════════════════════
cleaning_params = dict(
    const_threshold=1e-6,
    missing_threshold=0.25,
    remove_duplicates=True,
    corr_threshold=0.95,
    add_indicator=True,
    indicator_threshold=0.01,
    imputation_method='median',
)

# ═══════════════════════════════════════════════════════════════
# [전처리] 이상치 처리
# method: 'winsorize' | 'iqr_clip' | 'grubbs' | 'lot_local' | 'none'
# ═══════════════════════════════════════════════════════════════
outlier_params = dict(
    method='winsorize',
    # --- winsorize / lot_local 공통 ---
    lower_pct=0.01,
    upper_pct=0.995,
    # --- iqr_clip 전용 ---
    iqr_multiplier=1.5,            # IQR 배수 (1.5=표준, 3.0=극단값만)
    # --- grubbs 전용 ---
    # grubbs_alpha=0.05,           # 유의수준 (낮을수록 보수적)
    # grubbs_max_rounds=5,         # 반복 검정 횟수
    # --- lot_local 전용 ---
    # lot_run_id_col='run_wf_xy',  # 로트 구분 컬럼
    # lot_min_size=10,             # 최소 로트 크기
)

# ═══════════════════════════════════════════════════════════════
# [전처리] Feature Selection
# ═══════════════════════════════════════════════════════════════
selection_params = dict(
    methods=['boruta', 'lgbm_importance', 'null_importance', 'permutation'],
    min_votes=3,
    sample_n=None,
    boruta_params=dict(max_iter=100, max_depth=5, perc=80),
    lgbm_params=dict(threshold=5),
    null_params=dict(n_runs=10, threshold=2.0),
    rfe_params=dict(n_features_to_select=100, step=50),
    perm_params=dict(threshold=5, n_repeats=5),
    mi_params=dict(threshold=0),
)

# ═══════════════════════════════════════════════════════════════
# [샘플링]
# ═══════════════════════════════════════════════════════════════
sampling_params = dict(
    use_sampling=True,
    sample_frac=0.2,
)

# ═══════════════════════════════════════════════════════════════
# [집계] Die→Unit
# ═══════════════════════════════════════════════════════════════
agg_params = dict(
    agg_funcs=['mean', 'std', 'cv', 'range', 'min', 'max', 'median'],
    proba_agg='mean',
)

# ═══════════════════════════════════════════════════════════════
# [Stage 1] 분류기
# model: 'lgbm' | 'xgb' | 'catboost' | 'rf' | 'et'
# imbalance: 'none' | 'scale_pos_weight' | 'smote'
# ═══════════════════════════════════════════════════════════════
CLF_MODEL = 'lgbm'
clf_params = get_default_params(CLF_MODEL, 'clf')
# 커스텀 오버라이드:
# clf_params['n_estimators'] = 2000
# clf_params['learning_rate'] = 0.03

imbalance_params = dict(
    method='none',               # 'none' | 'scale_pos_weight' | 'smote'
    smote_params=None,           # SMOTE일 때: {'sampling_strategy': 0.5}
)

# ═══════════════════════════════════════════════════════════════
# [Stage 2] 회귀기
# model: 'lgbm' | 'xgb' | 'catboost' | 'rf' | 'et'
# objective (lgbm): 'regression' | 'poisson' | 'tweedie' | 'huber'
# ═══════════════════════════════════════════════════════════════
REG_MODEL = 'lgbm'
reg_params = get_default_params(REG_MODEL, 'reg')
# 커스텀 오버라이드:
# reg_params['objective'] = 'poisson'
# reg_params['objective'] = 'tweedie'
# reg_params['tweedie_variance_power'] = 1.5

# ═══════════════════════════════════════════════════════════════
# [OOF / Early Stopping]
# ═══════════════════════════════════════════════════════════════
cv_params = dict(
    n_folds=5,
    clf_early_stop=50,
    reg_early_stop=50,
)
LABEL_COL = 'label_bin'

# ═══════════════════════════════════════════════════════════════
# [후처리] CLF Threshold
# ═══════════════════════════════════════════════════════════════
threshold_params = dict(
    initial=0.5,           # 초기값 (튜닝 전)
    search_range=(0.0, 0.95),
    n_points=200,
)

# ═══════════════════════════════════════════════════════════════
# [다중 모델 비교] Stage 7-A
# ═══════════════════════════════════════════════════════════════
multi_model_params = dict(
    model_list=['lgbm', 'xgb', 'catboost', 'rf', 'et'],
)

# ═══════════════════════════════════════════════════════════════
# [HPO] Optuna
# ═══════════════════════════════════════════════════════════════
hpo_params = dict(
    clf_n_trials=50,
    reg_n_trials=100,
    # 고정 파라미터 (탐색에서 제외하고 싶은 값):
    reg_fixed={},          # 예: {'objective': 'poisson'}
    clf_fixed={},
)

# ═══════════════════════════════════════════════════════════════
# [앙상블]
# ═══════════════════════════════════════════════════════════════
ensemble_params = dict(
    voting=dict(
        optimize_weights=True,   # True: Optuna 최적화, False: RMSE 역수 비례
        n_trials=200,
    ),
    stacking=dict(
        meta_alpha=1.0,          # Ridge alpha
        use_features=False,      # True: OOF + unit feature, False: OOF만
        n_folds=5,
    ),
)

# ═══════════════════════════════════════════════════════════════
# 설정 요약 출력 + 중복 검사
# ═══════════════════════════════════════════════════════════════
CLF_THRESHOLD = threshold_params['initial']

print(f'실험번호: {EXP_ID} | {EXP_TYPE}')
print(f'CLF: {CLF_MODEL} | REG: {REG_MODEL} | 불균형: {imbalance_params["method"]}')
print(f'집계: {agg_params["agg_funcs"]}')
print(f'샘플링: {"ON" if sampling_params["use_sampling"] else "OFF"} '
      f'(frac={sampling_params["sample_frac"]})')
print(f'이상치: {outlier_params["method"]}')
print(f'회귀 objective: {reg_params.get("objective", "default")}')
print(f'스위치: multi_model={RUN_MULTI_MODEL} | threshold={RUN_THRESHOLD_TUNING} '
      f'| hpo={RUN_HPO} | voting={RUN_VOTING} | stacking={RUN_STACKING}')

download_from_drive(XLSX_GDRIVE_ID, JSON_GDRIVE_ID)
check_exp_id(EXP_ID)

# 모든 파라미터를 합쳐서 중복 검사
_all_model_params = {
    'clf_model': CLF_MODEL, 'clf': clf_params,
    'reg_model': REG_MODEL, 'reg': reg_params,
    'cv': cv_params,
    'imbalance': imbalance_params,
    'sampling': sampling_params,
}
check_duplicate_params(
    EXP_ID,
    cleaning_params=cleaning_params,
    outlier_params=outlier_params,
    feature_sel_params=selection_params,
    agg_params=agg_params,
    model_params=_all_model_params,
)
print('파라미터 중복 검사 통과')

## 2. 전처리 (클리닝 + 이상치)

In [ ]:
# --- Step 1: 클리닝 ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    **cleaning_params,
)


In [ ]:
# --- Step 2: 이상치 처리 ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    **outlier_params,
)


## 3. Die-level Health Merge + 라벨 생성 + Position 분리

- Health(unit-level)을 die-level로 merge
- 라벨 생성: 0 vs >0 (분류기용)
- Position 1~4로 분리


In [ ]:
# ============================================================
# Die-level로 Health(target) merge
# ============================================================
die_train = xs_train.merge(ys['train'], on=KEY_COL, how='left')
die_val = xs_val.merge(ys['validation'], on=KEY_COL, how='left')
die_test = xs_test.merge(ys['test'], on=KEY_COL, how='left')

assert die_train[TARGET_COL].notna().all(), 'train health NaN!'
assert die_val[TARGET_COL].notna().all(), 'val health NaN!'
assert die_test[TARGET_COL].notna().all(), 'test health NaN!'
print(f'Die-level merge: train={die_train.shape}, val={die_val.shape}, test={die_test.shape}')

# ============================================================
# 샘플링 (unit 기준, train만)
# ============================================================
USE_SAMPLING = sampling_params['use_sampling']
SAMPLE_FRAC = sampling_params['sample_frac']

if USE_SAMPLING:
    all_units = die_train[KEY_COL].drop_duplicates()
    sampled_units = all_units.sample(frac=SAMPLE_FRAC, random_state=SEED)
    n_before = len(die_train)
    die_train = die_train[die_train[KEY_COL].isin(sampled_units)].reset_index(drop=True)
    print(f"\n샘플링 ON (frac={SAMPLE_FRAC})")
    print(f'  Unit: {len(all_units):,} -> {len(sampled_units):,}')
    print(f'  Die: {n_before:,} -> {len(die_train):,}')
else:
    print("\n샘플링 OFF -> 전체 데이터 사용")

# --- 라벨 생성: 0 vs >0 ---
for df in [die_train, die_val, die_test]:
    df[LABEL_COL] = (df[TARGET_COL] > 0).astype(int)

print("\n라벨 분포 (train):")
dist = die_train[LABEL_COL].value_counts().sort_index()
for k, v in dist.items():
    pct = v / len(die_train) * 100
    print(f'  {k}: {v:,} ({pct:.1f}%)')

# ============================================================
# Position별 분리
# ============================================================
positions = sorted(die_train[POSITION_COL].unique())
feat_cols_clean = clean_cols
AGG_FUNCS = agg_params['agg_funcs']

pos_data = {}
for pos in positions:
    pos_data[pos] = {
        'train': die_train[die_train[POSITION_COL] == pos].reset_index(drop=True),
        'val':   die_val[die_val[POSITION_COL] == pos].reset_index(drop=True),
        'test':  die_test[die_test[POSITION_COL] == pos].reset_index(drop=True),
    }
    n = {k: len(v) for k, v in pos_data[pos].items()}
    print(f'Position {pos}: train={n["train"]:,}, val={n["val"]:,}, test={n["test"]:,}')

print(f'\nClean feature 수: {len(feat_cols_clean)}개')

## 4. Stage 1: Position별 이진분류 (OOF)

- Position별로 분류기를 K-Fold OOF로 학습
- 각 die에 대한 P(Y>0) 확률을 산출
- 불균형 처리: `IMBALANCE_METHOD`로 제어


In [ ]:
N_FOLDS = cv_params['n_folds']
CLF_EARLY_STOP = cv_params['clf_early_stop']
REG_EARLY_STOP = cv_params['reg_early_stop']

print(f'Stage 1: Position별 이진분류 (OOF) - {CLF_MODEL}')
print('=' * 50)
clf_result = run_classification(
    pos_data, feat_cols_clean, clf_params,
    model_name=CLF_MODEL,
    n_folds=N_FOLDS,
    early_stop=CLF_EARLY_STOP,
    label_col=LABEL_COL,
    clf_threshold=CLF_THRESHOLD,
    imbalance_method=imbalance_params['method'],
    imbalance_params=imbalance_params.get('smote_params'),
)

## 5. Die -> Unit 집계

- 집계 함수: mean, std, cv, range, min, max, median
- Position별 피벗 feature 추가
- 분류 확률 집계 (clf_proba_mean)


In [ ]:
def aggregate_die_to_unit(pos_data, feat_cols, clf_result, agg_funcs=AGG_FUNCS):
    """Die-level -> Unit-level 집계"""
    results = {}

    for split_name in ['train', 'val', 'test']:
        die_frames = []
        for pos in sorted(pos_data.keys()):
            df = pos_data[pos][split_name].copy()
            df['clf_proba'] = clf_result[pos][f'{split_name}_proba']
            die_frames.append(df)

        die_all = pd.concat(die_frames, ignore_index=True)

        # --- WT feature 집계 ---
        agg_dict = {}
        grp = die_all.groupby(KEY_COL)[feat_cols]

        for func in agg_funcs:
            if func == 'cv':
                cv_df = grp.std() / grp.mean().abs().clip(lower=1.0)
                cv_df.columns = [f'{c}_cv' for c in feat_cols]
                agg_dict['cv'] = cv_df
            elif func == 'range':
                range_df = grp.max() - grp.min()
                range_df.columns = [f'{c}_range' for c in feat_cols]
                agg_dict['range'] = range_df
            else:
                grp_df = grp.agg(func)
                grp_df.columns = [f'{c}_{func}' for c in feat_cols]
                agg_dict[func] = grp_df

        unit_features = pd.concat(agg_dict.values(), axis=1)

        # --- Position별 피벗 ---
        for pos in sorted(pos_data.keys()):
            pos_mask = die_all[POSITION_COL] == pos
            pos_df = die_all.loc[pos_mask].set_index(KEY_COL)[feat_cols]
            pos_df.columns = [f'{c}_pos{pos}' for c in feat_cols]
            unit_features = unit_features.join(pos_df)

        # --- 분류 확률 집계 ---
        proba_mean = die_all.groupby(KEY_COL)['clf_proba'].mean()
        proba_mean.name = 'clf_proba_mean'
        unit_features = unit_features.join(proba_mean)

        # --- health (target) ---
        health = die_all.groupby(KEY_COL)[TARGET_COL].first()
        unit_features = unit_features.join(health)

        unit_features = unit_features.reset_index()
        results[split_name] = unit_features

    return results


print('Die -> Unit 집계')
print('=' * 50)
unit_data = aggregate_die_to_unit(pos_data, feat_cols_clean, clf_result)

unit_feat_cols = [c for c in unit_data['train'].columns
                  if c not in [KEY_COL, TARGET_COL]]

for sp in ['train', 'val', 'test']:
    print(f'  {sp}: {unit_data[sp].shape}')
print(f'  Unit-level feature 수: {len(unit_feat_cols)}개')


## 5.5 Feature Selection (Unit-level)

- 분산 0 제거 -> 투표 기반 선택 (Boruta + LightGBM + Null Importance + Permutation)


In [ ]:
# --- 사전 정리: 분산 0인 unit feature 제거 ---
var = unit_data['train'][unit_feat_cols].var()
nonzero_cols = var[var > 0].index.tolist()
zero_var_removed = len(unit_feat_cols) - len(nonzero_cols)
print(f'Zero-variance unit feature 제거: {zero_var_removed}개')
print(f'Feature Selection 입력: {len(nonzero_cols)}개')

# --- Feature Selection ---
selected_cols, sel_report = run_feature_selection(
    X_train=unit_data['train'],
    y_train=unit_data['train'][TARGET_COL],
    feat_cols=nonzero_cols,
    **selection_params,
)

n_before = len(unit_feat_cols)
unit_feat_cols = selected_cols
print(f'\nFeature Selection 결과: {n_before} -> {len(unit_feat_cols)}개')

if sel_report.get('vote_df') is not None:
    vote_df = sel_report['vote_df']
    print('\n--- 투표 현황 ---')
    for v in sorted(vote_df['votes'].unique(), reverse=True):
        cnt = (vote_df['votes'] == v).sum()
        print(f'  {v}표: {cnt}개')
    print('\n상위 10 features (투표순):')
    print(vote_df.head(10).to_string(index=False))


## 6. Stage 2: Unit-level 회귀

**방식 A - True Two-Stage**: Y>0 서브셋에서 회귀 -> P(Y>0) x E[Y|Y>0]

**방식 B - Single Regression**: 전체 데이터 회귀, clf_proba_mean이 feature


In [ ]:
# ============================================================
# 두 방식 동시 실행 + 비교
# ============================================================
y_val = unit_data['val'][TARGET_COL].values
y_test = unit_data['test'][TARGET_COL].values

print(f'방식 A: True Two-Stage ({REG_MODEL})')
print('=' * 50)
result_twostage = run_twostage_regression(
    unit_data, unit_feat_cols, reg_params,
    model_name=REG_MODEL, n_folds=N_FOLDS, early_stop=REG_EARLY_STOP,
)

print(f'\n방식 B: Single Regression ({REG_MODEL})')
print('=' * 50)
result_single = run_single_regression(
    unit_data, unit_feat_cols, reg_params,
    model_name=REG_MODEL, n_folds=N_FOLDS, early_stop=REG_EARLY_STOP,
)

# --- 비교 ---
print(f'\n{"=" * 60}')
print('최종 RMSE 비교')
print(f'{"=" * 60}')
print('\n[True Two-Stage]')
ts_val_rmse = evaluate(y_val, result_twostage['val_pred'], label='val')
ts_test_rmse = evaluate(y_test, result_twostage['test_pred'], label='test')

print('\n[Single Regression]')
sg_val_rmse = evaluate(y_val, result_single['val_pred'], label='val')
sg_test_rmse = evaluate(y_test, result_single['test_pred'], label='test')

if ts_val_rmse <= sg_val_rmse:
    best_method = 'True Two-Stage'
    best_val_rmse = ts_val_rmse
    best_test_rmse = ts_test_rmse
    best_result = result_twostage
else:
    best_method = 'Single Regression'
    best_val_rmse = sg_val_rmse
    best_test_rmse = sg_test_rmse
    best_result = result_single

print(f'\n>>> 최적 방식: {best_method} (val RMSE: {best_val_rmse:.6f})')


## 6.5 다중 모델 비교 (Stage 7-A)

5종 트리 모델을 동일 조건에서 비교하여 최적 모델을 선별합니다.

> `RUN_MULTI_MODEL = False`이면 이 셀은 스킵됩니다.


In [ ]:
if RUN_MULTI_MODEL:
    print('다중 모델 비교 (Stage 7-A)')
    print('=' * 60)

    model_configs = {}
    for m in multi_model_params['model_list']:
        model_configs[m] = {'params': get_default_params(m, 'reg')}

    # 현재 reg_params가 커스텀이면 해당 모델은 커스텀으로 덮어쓰기
    model_configs[REG_MODEL] = {'params': reg_params}

    multi_results, multi_comparison = run_multi_model_comparison(
        unit_data, unit_feat_cols, model_configs,
        mode='twostage' if best_method == 'True Two-Stage' else 'single',
        n_folds=N_FOLDS, early_stop=REG_EARLY_STOP,
    )

    # 최적 모델 갱신
    best_multi_model = multi_comparison.iloc[0]['model']
    if multi_results[best_multi_model]['val_rmse'] < best_val_rmse:
        best_val_rmse = multi_results[best_multi_model]['val_rmse']
        best_result = multi_results[best_multi_model]
        best_test_rmse = evaluate(
            y_test, best_result['test_pred'], label='best test'
        )
        print(f'\n>>> 최적 모델 갱신: {best_multi_model} '
              f'(val RMSE: {best_val_rmse:.6f})')
else:
    print('다중 모델 비교 스킵 (RUN_MULTI_MODEL=False)')
    multi_results = None

## 7. CLF Threshold 튜닝

P(Y>0) < threshold인 샘플의 예측을 0으로 마스킹하여 RMSE를 최소화하는 최적 threshold를 탐색합니다.


In [ ]:
if RUN_THRESHOLD_TUNING:
    from modules.threshold import optimize_threshold

    best_thr, thr_rmse, thr_results = optimize_threshold(
        proba=unit_data['val']['clf_proba_mean'].values,
        reg_pred=best_result['val_pred'],
        y_true=y_val,
        search_range=threshold_params['search_range'],
        n_points=threshold_params['n_points'],
    )

    print(f'\n기존 (threshold=없음): val RMSE = {best_val_rmse:.6f}')
    print(f'최적 threshold={best_thr:.3f}: val RMSE = {thr_rmse:.6f}')

    if thr_rmse < best_val_rmse:
        print(f'-> Threshold 적용으로 RMSE {best_val_rmse - thr_rmse:.6f} 개선!')
        CLF_THRESHOLD = best_thr
        best_val_rmse = thr_rmse
        # test에도 동일 threshold 적용
        from modules.threshold import apply_threshold
        masked_test = apply_threshold(
            unit_data['test']['clf_proba_mean'].values,
            best_result['test_pred'], CLF_THRESHOLD,
        )
        best_test_rmse = evaluate(y_test, masked_test, label='test (masked)')
    else:
        print('-> Threshold 미적용 (개선 없음)')
else:
    print('CLF Threshold 튜닝 스킵 (RUN_THRESHOLD_TUNING=False)')

## 7.5 HPO (Optuna)

Optuna로 분류기/회귀기 하이퍼파라미터를 자동 탐색합니다.

> `RUN_HPO = False`이면 이 셀은 스킵됩니다. HPO는 시간이 오래 걸리므로 필요할 때만 켜세요.

In [ ]:
if RUN_HPO:
    from modules.hpo import optimize_clf, optimize_reg

    # --- CLF HPO ---
    print(f'CLF HPO ({CLF_MODEL}, {hpo_params["clf_n_trials"]} trials)')
    print('=' * 50)
    best_clf_params, clf_study = optimize_clf(
        pos_data, feat_cols_clean,
        model_name=CLF_MODEL,
        n_trials=hpo_params['clf_n_trials'],
        n_folds=N_FOLDS,
        early_stop=CLF_EARLY_STOP,
        label_col=LABEL_COL,
        hpo_params=hpo_params.get('clf_fixed', {}),
    )
    clf_params = best_clf_params
    print(f'\nCLF 최적 파라미터 적용 완료')

    # --- REG HPO ---
    print(f'\nREG HPO ({REG_MODEL}, {hpo_params["reg_n_trials"]} trials)')
    print('=' * 50)
    best_reg_params, reg_study = optimize_reg(
        unit_data, unit_feat_cols,
        model_name=REG_MODEL,
        mode='twostage' if best_method == 'True Two-Stage' else 'single',
        n_trials=hpo_params['reg_n_trials'],
        n_folds=N_FOLDS,
        early_stop=REG_EARLY_STOP,
        hpo_params=hpo_params.get('reg_fixed', {}),
    )
    reg_params = best_reg_params
    print(f'\nREG 최적 파라미터 적용 완료')

    # --- HPO 결과로 Stage 2 재실행 ---
    print(f'\n{"=" * 50}')
    print('HPO 최적 파라미터로 Stage 2 재실행')
    print(f'{"=" * 50}')

    result_twostage_hpo = run_twostage_regression(
        unit_data, unit_feat_cols, reg_params,
        model_name=REG_MODEL, n_folds=N_FOLDS, early_stop=REG_EARLY_STOP,
    )
    hpo_val_rmse = evaluate(y_val, result_twostage_hpo['val_pred'], label='HPO val')
    hpo_test_rmse = evaluate(y_test, result_twostage_hpo['test_pred'], label='HPO test')

    if hpo_val_rmse < best_val_rmse:
        best_val_rmse = hpo_val_rmse
        best_test_rmse = hpo_test_rmse
        best_result = result_twostage_hpo
        best_method = f'HPO {REG_MODEL}'
        print(f'\n>>> HPO 개선! (val RMSE: {best_val_rmse:.6f})')
    else:
        print(f'\n>>> HPO 미개선 (기존: {best_val_rmse:.6f})')
else:
    print('HPO 스킵 (RUN_HPO=False)')

## 8. 앙상블 (보팅 + 스태킹)

다중 모델 비교 결과를 활용하여 앙상블을 구성합니다.

> `RUN_ENSEMBLE = False (= RUN_VOTING and RUN_STACKING 모두 False)`이면 이 셀은 스킵됩니다.


In [ ]:
if (RUN_VOTING or RUN_STACKING) and multi_results is not None:
    from modules.ensemble import run_voting, run_stacking

    print('앙상블')
    print('=' * 60)

    voting_result = None
    stacking_result = None

    # --- 보팅 ---
    if RUN_VOTING:
        voting_result = run_voting(
            multi_results, y_val, y_test,
            optimize_weights=ensemble_params['voting']['optimize_weights'],
            n_trials=ensemble_params['voting']['n_trials'],
        )
    else:
        print('보팅 스킵 (RUN_VOTING=False)')

    # --- 스태킹 ---
    if RUN_STACKING:
        stacking_result = run_stacking(
            multi_results, unit_data, unit_feat_cols,
            y_val, y_test,
            stacking_params=ensemble_params['stacking'],
        )
    else:
        print('스태킹 스킵 (RUN_STACKING=False)')

    # --- 앙상블 결과가 best보다 나으면 갱신 ---
    for ens_name, ens_result in [('voting', voting_result), ('stacking', stacking_result)]:
        if ens_result is not None and ens_result['val_rmse'] < best_val_rmse:
            best_val_rmse = ens_result['val_rmse']
            best_test_rmse = ens_result['test_rmse']
            best_result = ens_result
            best_method = ens_name
            print(f'\n>>> 최적 갱신: {ens_name} (val RMSE: {best_val_rmse:.6f})')
else:
    if not (RUN_VOTING or RUN_STACKING):
        print('앙상블 스킵 (RUN_VOTING=False, RUN_STACKING=False)')
    else:
        print('앙상블 스킵 (다중 모델 비교 결과 없음)')

## 9. 시각화

In [ ]:
# TODO: 시각화 코드 추가 예정
# (실험 로그는 Cell 28에서 실행)


## 10. 실험 로그

In [ ]:
# ============================================================
# 실험 로그 — 모든 파라미터 기록
# ============================================================
log_experiment(
    exp_id=EXP_ID,
    exp_type=EXP_TYPE,
    best_model=f'{REG_MODEL} ({best_method})',
    val_rmse=best_val_rmse,
    test_rmse=best_test_rmse,
    n_features=len(unit_feat_cols),
    memo=EXP_MEMO,
    cleaning_params=cleaning_params,
    outlier_params=outlier_params,
    feature_sel_params=selection_params,
    agg_params=agg_params,
    model_params={
        # 스위치
        'switches': {
            'run_multi_model': RUN_MULTI_MODEL,
            'run_threshold_tuning': RUN_THRESHOLD_TUNING,
            'run_hpo': RUN_HPO,
            'run_voting': RUN_VOTING,
            'run_stacking': RUN_STACKING,
        },
        # 모델
        'clf_model': CLF_MODEL, 'clf': clf_params,
        'reg_model': REG_MODEL, 'reg': reg_params,
        'imbalance': imbalance_params,
        # CV
        'cv': cv_params,
        # 샘플링
        'sampling': sampling_params,
        # 후처리
        'threshold': threshold_params,
        'clf_threshold_final': CLF_THRESHOLD,
        # 다중 모델
        'multi_model': multi_model_params if RUN_MULTI_MODEL else None,
        # HPO
        'hpo': hpo_params if RUN_HPO else None,
        # 앙상블
        'ensemble': ensemble_params if (RUN_VOTING or RUN_STACKING) else None,
    },
    feature_cols=unit_feat_cols,
    xlsx_gdrive_id=XLSX_GDRIVE_ID,
    json_gdrive_id=JSON_GDRIVE_ID,
)